# Solutions 12: Four Follow-On Studies

This notebook solves the four exercises of Lab 12. The exercises are study designs, not
runs: each one asks for a second study forked from the capstone's protocol. Accordingly,
every solution's deliverable executes live (a modified pre-registered protocol, frozen and
hashed exactly like the lab's Part A-1, with its power reasoning and its decision rules
asserted), and the twelve-to-eighteen training runs each protocol describes are gated behind
`RUN_STUDY = False`. Attempt the exercises yourself before reading this file: designing a
study is the skill the capstone certifies, and reading a finished protocol teaches less than
drafting one and discovering what it forgot.

A reminder of the machinery being reused, in one line each. A pre-registered protocol is the
full plan (hypotheses, arms, seeds, metrics, stopping and exclusion rules) written down and
frozen by hashing before any run starts. The hash is the commitment: change one character and
the hash changes visibly. Power is a design's ability to detect the effect it claims to
test above its own seed noise, and Lab 05's measured seed spread is the noise estimate this
course uses. Each solution below states the hypotheses, the arms, the power implications, and
the concrete result that would decide the question either way.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, hashlib
sys.path.insert(0, "../code")

import torch
from kd_pipeline import set_seed_everywhere, config_fingerprint

RUN_STUDY = False        # <-- flip on the training box
SEED0 = 2000             # follow-on study seeds: disjoint from the labs (17) and the
set_seed_everywhere(SEED0)   # capstone (1000-1002), so no run can be silently reused
os.makedirs("../runs/lab12", exist_ok=True)

def freeze(protocol, name):
    # The lab's Part A-1 freeze, as a function: dump, hash, write, reload, verify.
    frozen = json.dumps(protocol, sort_keys=True, indent=2)
    h = hashlib.sha256(frozen.encode()).hexdigest()[:16]
    path = f"../runs/lab12/protocol_{name}_{h}.json"
    open(path, "w").write(frozen)
    reloaded = open(path).read()
    assert hashlib.sha256(reloaded.encode()).hexdigest()[:16] == h, "freeze must verify"
    print(f"protocol frozen: {path}\ncommitment hash: {h}")
    return h

def seed_spread():
    # Lab 05's measured within-arm seed range, or the course prior if the file
    # is absent on this machine (same fallback the lab itself uses).
    try:
        rows = json.load(open("../runs/lab05/results.json"))
        by_arm = {}
        for r in rows:
            by_arm.setdefault(r["beta"], []).append(r["agreement"])
        return max(max(v) - min(v) for v in by_arm.values() if len(v) > 1), "lab05 measured"
    except FileNotFoundError:
        return 0.015, "course prior (run Lab 05 to replace me)"

SPREAD, SPREAD_SRC = seed_spread()
print(f"seed-noise input for every power check below: {SPREAD:.3f} ({SPREAD_SRC})")

seed-noise input for every power check below: 0.015 (course prior (run Lab 05 to replace me))


## Exercise 1: The beta interaction

**The exercise, restated.** Cross Lab 05 with the capstone's design: does the off-policy
versus on-policy gap depend on the choice of divergence in the loss? The literature says
on-policy training tolerates reverse KL better.

**The approach.** This is a factorial design, meaning every combination of two factors is
run: training policy (off-policy cached, on-policy) crossed with the divergence knob beta
(0.0 for forward KL, 0.5 for symmetric JSD, 1.0 for reverse KL, in the TRL convention Lab 01
verified numerically). One student size keeps the run count sane: 2 x 3 arms x 3 seeds is 18
runs. The quantity under test is an *interaction*: not "is on-policy better" (the capstone
asked that) and not "which beta is best" (Lab 05 asked that), but whether the on-policy
advantage *changes* as beta moves, which is a difference of differences. That last phrase is
the power trap the protocol must handle explicitly: a difference of differences stacks four
cell means, so its noise is roughly twice a single gap's noise, which means the registered
rule must demand a bigger effect before claiming anything. The power check below therefore
doubles the seed-spread floor before comparing it against the smallest interaction the study
would call interesting, and the assert fails the design loudly if 3 seeds cannot resolve it,
which is the moment to add seeds, before the 18 runs, not after.

In [2]:
P1 = {
  "title": "Does the off/on-policy gap depend on the divergence? (capstone x lab05)",
  "forked_from": "capstone protocol (see ../runs/lab12/protocol_*.json, lab 12 Part A-1)",
  "hypotheses": {
    "H1": "The on-policy minus off-policy gap on the primary metric is larger at "
          "beta=1.0 (reverse KL) than at beta=0.0 (forward KL).",
    "H0": "The gap does not vary with beta beyond the interaction noise band.",
  },
  "arms": {f"{pol}-beta{b}": {"recipe": ("lab04-cached-topk64" if pol == "off"
                                          else "lab07-gkd-warmstart"),
                              "beta": b, "student": "SmolLM2-135M"}
           for pol in ("off", "on") for b in (0.0, 0.5, 1.0)},
  "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
  "seeds": [SEED0, SEED0 + 1, SEED0 + 2],
  "primary_metric": "teacher-scored mean logprob of student rollouts, 128 held-out prompts",
  "compute_matching": "capstone A-2 rule: matched token-passes per arm, verified from logs",
  "stopping_rule": "fixed token-pass budget; no early stopping on metrics",
  "exclusion_rule": "capstone rule inherited: collapsed runs kept and marked; a cell is "
                    "excluded only if >=2/3 seeds trip the monitors (reported as fragile)",
  "analysis": "interaction = (mean_on - mean_off at beta=1.0) - (mean_on - mean_off at "
              "beta=0.0); claimed only if |interaction| > 2 x max within-cell seed range",
}
h1 = freeze(P1, "ex1-beta-interaction")

MDE_INTERACTION = 2 * SPREAD          # a difference of differences doubles the noise floor
INTERESTING = 0.05                    # smallest interaction H1 would care about, in metric units
n_runs = len(P1["arms"]) * len(P1["seeds"])
print(f"runs: {n_runs} | interaction MDE: {MDE_INTERACTION:.3f} | "
      f"interesting threshold: {INTERESTING:.3f}")
assert INTERESTING > MDE_INTERACTION, "3 seeds cannot resolve the interaction; add seeds NOW"
assert len(P1["arms"]) == 6 and n_runs == 18
if not RUN_STUDY:
    print("RUN_STUDY=False: 18 runs gated; the design above is the deliverable")

protocol frozen: ../runs/lab12/protocol_ex1-beta-interaction_096f28ff3a7665fa.json
commitment hash: 096f28ff3a7665fa
runs: 18 | interaction MDE: 0.030 | interesting threshold: 0.050
RUN_STUDY=False: 18 runs gated; the design above is the deliverable


**Interpretation.** The protocol froze and verified, and the power check passed with
the doubled floor: the registered rule can certify an interaction of 0.05 in the primary
metric against a noise floor of 0.030 (twice the 0.015 seed spread), so the design is capable
of answering its own question at 3 seeds, with little margin to spare. That thin margin is
itself a finding worth registering: if Lab 05's measured spread came out larger than the
prior on your machine, the assert fails and the honest fixes are five seeds per cell or a
larger interesting-effect threshold, decided before results exist.

What would decide the question. H1 is supported if the on-policy advantage at reverse KL
exceeds the advantage at forward KL by more than the doubled noise band; that outcome matches
the literature's mechanism (reverse KL is mode seeking, and mode seeking on the student's own
rollouts cannot hide from its own mistakes the way it can on teacher-forced text). H1 is
refuted, interestingly, if the interaction lands inside the band or reversed: that would say
the capstone's off/on conclusion transfers across divergences, and the divergence choice can
be made on Lab 05's single-axis evidence alone. Both outcomes are publishable under the
frozen hash, which is the point of freezing it.

## Exercise 2: Scale one axis

**The exercise, restated.** Re-run the capstone with the 1.7B teacher swapped for a served
8B-class teacher, using Lab 08's serving topology. Which conclusions survive a 5x bigger
teacher?

**The approach.** The design principle is in the exercise's own title: scale exactly one
axis, and keep every other capstone choice frozen, because a second changed variable would
make any changed conclusion unattributable. The arms are the capstone's four (off-policy and
on-policy, at 360M and 135M students), rerun against the big teacher; the capstone's own
results serve as the small-teacher half of the comparison, which halves the new compute. The
genuinely new design work is in the compute-matching clause, and the protocol spells it out
rather than inheriting it blindly: the capstone's token-pass currency priced all passes as
equal, which was defensible when one teacher's passes appeared in every arm, but an 8B pass
is not a 1.7B pass. The forked rule matches *student-side* token-passes across arms (that is
the quantity the student's learning is bought with) and reports teacher-side passes as a
separate, explicitly not-matched cost column, so a reader can see what the bigger teacher
actually cost. The power reasoning reuses the capstone's single-gap floor, since the headline
questions are plain gaps, not interactions.

In [3]:
P2 = {
  "title": "Which capstone conclusions survive a 5x teacher? (one axis scaled)",
  "forked_from": "capstone protocol; capstone results reused as the small-teacher arms",
  "hypotheses": {
    "H1": "The off/on-policy ordering on the primary metric is unchanged under the "
          "8B-class teacher (direction survives).",
    "H2": "Both policies gain from the bigger teacher, and off-policy gains at least as "
          "much (its teacher cost is prepaid into the cache; lab 06 economics).",
    "H0": "Any change sits inside the seed-noise band.",
  },
  "arms": {f"{pol}-{size}": {"recipe": ("lab04-cached-topk64" if pol == "off"
                                         else "lab07-gkd-warmstart"),
                             "student": f"SmolLM2-{size}"}
           for pol in ("off", "on") for size in ("360M", "135M")},
  "teacher": "8B-class instruct model, served via lab 08 topology (one server, "
             "students as clients); exact checkpoint pinned by hash at run time",
  "seeds": [SEED0 + 10, SEED0 + 11, SEED0 + 12],
  "primary_metric": "teacher-scored mean logprob of student rollouts, 128 held-out prompts, "
                    "scored by BOTH teachers so scale-of-scorer is visible",
  "compute_matching": "match student-side token-passes to the capstone's budget exactly; "
                      "teacher-side passes reported unmatched in their own cost column; "
                      "server throughput logged so queueing cannot silently starve an arm",
  "stopping_rule": "fixed student-side token-pass budget; no early stopping on metrics",
  "exclusion_rule": "capstone rule inherited verbatim",
  "analysis": "per capstone rule: effect claimed only if arm-mean gap > max within-arm "
              "seed range; H1 verdict is the sign of the off/on gap per student size",
}
h2 = freeze(P2, "ex2-teacher-scale")

MDE = SPREAD
print(f"runs: {len(P2['arms']) * len(P2['seeds'])} new (capstone supplies the other 12)")
print(f"single-gap MDE: {MDE:.3f}; H1 is a sign question, decided by the capstone rule")
assert "student-side" in P2["compute_matching"], \
    "the forked matching rule must name the currency it matches"
assert P2["seeds"] != P1["seeds"], "fresh seeds: no run may be silently reused across studies"
assert "BOTH teachers" in P2["primary_metric"], \
    "a scale study must not let the scorer's scale hide inside the metric"
if not RUN_STUDY:
    print("RUN_STUDY=False: 12 runs gated; the design above is the deliverable")

protocol frozen: ../runs/lab12/protocol_ex2-teacher-scale_0b416e723be537d7.json
commitment hash: 0b416e723be537d7
runs: 12 new (capstone supplies the other 12)
single-gap MDE: 0.015; H1 is a sign question, decided by the capstone rule
RUN_STUDY=False: 12 runs gated; the design above is the deliverable


**Interpretation.** The protocol froze with the three design guards asserted, and each
guard exists because of a specific way this study could quietly lie. The compute-matching
guard: if teacher passes were matched instead of student passes, the 8B arms would get far
fewer student steps and the study would "find" that big teachers hurt. The scorer guard: the
primary metric is teacher-scored, so swapping the teacher swaps the judge; scoring every
rollout with both teachers turns a confounded metric into a visible two-column comparison,
and the honest headline is the gap measured by the *same* scorer across teacher arms. The
fresh-seeds guard prevents the tempting shortcut of reusing capstone runs under new names,
which would correlate the halves of the comparison.

What would decide it. H1 survives if the off/on ordering keeps its sign per student size
under the capstone's gap-versus-seed-range rule; it fails informatively if the sign flips for
the 135M student only, which is the outcome Lab 07's exposure-bias reasoning would predict if
a sharper big-teacher distribution punishes weak students harder on their own rollouts. H2 is
decided by the two gain columns; off-policy gaining more supports the Lab 06 economics
reading (the cache amortizes the expensive teacher), while on-policy gaining more says the
big teacher's value is in scoring live student behavior, not in richer cached targets. The
expected shape, for calibration rather than commitment: H1 survives, H2 lands close to even,
and the unmatched teacher-cost column shows the 8B study costing several times the capstone,
which is the number a manager will actually ask for.

## Exercise 3: The sequencing curve

**The exercise, restated.** Between pure off-policy and warm-started on-policy lies a
schedule: spend 0%, 25%, 50%, or 75% of the budget off-policy first, then switch to
on-policy. Map the curve, find the knee (the point where more off-policy time stops
helping), and compare with Lab 07's cold-start findings.

**The approach.** Four arms, one per split fraction, three seeds, one student. The design
work that makes this study auditable is the budget accounting inside each arm: a split arm
spends `frac` of the token-pass budget at the off-policy per-step price and the remainder at
the on-policy per-step price (the capstone's A-2 prices: roughly 2 passes per token
off-policy, roughly 4 on-policy, so the phases have different step counts by construction).
The protocol freezes the per-arm step schedule, computed from the prices, and the live cell
*verifies the accounting mechanically*: it recomputes every arm's total token-passes from its
frozen step counts and asserts they all equal the budget within one step's worth of rounding.
That assert is this exercise's version of "matched is where studies quietly cheat": any
error in the split arithmetic would hand some fraction its own private budget increase, and
the curve's knee would be an artifact. The knee itself gets a registered definition, because
"find the knee" by eye is post-hoc: the knee is the smallest fraction whose improvement over
the next-smaller fraction is below the seed-noise floor.

In [4]:
SEQ, ROLL, BATCH = 384, 128, 8
COST_OFF = 2 * SEQ * BATCH                       # passes per optimizer step, capstone A-2
COST_ON = (ROLL + 2 * (SEQ + ROLL) + (SEQ + ROLL)) * BATCH   # gen + train + teacher score
BUDGET = 3.0e9

arms = {}
for frac in (0.00, 0.25, 0.50, 0.75):
    steps_off = int(frac * BUDGET / COST_OFF)
    steps_on = int((1 - frac) * BUDGET / COST_ON)
    arms[f"off{int(frac*100):02d}"] = {"frac_offpolicy": frac,
                                        "steps_off": steps_off, "steps_on": steps_on}

P3 = {
  "title": "The off-then-on sequencing curve: where does the knee sit?",
  "forked_from": "capstone protocol; arms replace the 2x2 with a 1x4 schedule sweep",
  "hypotheses": {
    "H1": "Quality on the primary metric rises from frac=0 to some interior fraction, "
          "then flattens: a knee exists strictly inside (0, 0.75].",
    "H0": "The curve is flat within seed noise (sequencing does not matter at this budget).",
  },
  "arms": arms, "student": "SmolLM2-135M",
  "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
  "seeds": [SEED0 + 20, SEED0 + 21, SEED0 + 22],
  "primary_metric": "teacher-scored mean logprob of student rollouts, 128 held-out prompts",
  "budget_token_passes": BUDGET,
  "step_prices": {"off_per_step": COST_OFF, "on_per_step": COST_ON},
  "knee_rule": "smallest frac whose gain over the next-smaller frac is < max within-arm "
               "seed range; registered here so the knee cannot be chosen by eye",
  "stopping_rule": "each phase runs its frozen step count exactly; no metric-based switching",
  "exclusion_rule": "capstone rule inherited verbatim",
}
h3 = freeze(P3, "ex3-sequencing")

print(f"\n{'arm':>7} {'steps off':>10} {'steps on':>9} {'total passes':>14}")
totals = []
for name, a in P3["arms"].items():
    total = a["steps_off"] * COST_OFF + a["steps_on"] * COST_ON
    totals.append(total)
    print(f"{name:>7} {a['steps_off']:>10,} {a['steps_on']:>9,} {total:>14,.0f}")
tol = max(COST_OFF, COST_ON)                     # one step's worth of integer rounding
assert all(abs(t - BUDGET) <= tol for t in totals), \
    "every arm must spend the same budget to within one step of rounding"
assert max(totals) - min(totals) <= 2 * tol, "no arm may quietly out-spend another"
print(f"\naccounting verified: all four arms within one step ({tol:,} passes) of "
      f"{BUDGET:.1e}; the knee, once measured, cannot be a budget artifact")
if not RUN_STUDY:
    print("RUN_STUDY=False: 12 runs gated; the frozen schedule above is the deliverable")

protocol frozen: ../runs/lab12/protocol_ex3-sequencing_1ae6b48856610e1a.json
commitment hash: 1ae6b48856610e1a

    arm  steps off  steps on   total passes
  off00          0   225,360  2,999,992,320
  off25    122,070   169,020  2,999,992,320
  off50    244,140   112,680  2,999,992,320
  off75    366,210    56,340  2,999,992,320

accounting verified: all four arms within one step (13,312 passes) of 3.0e+09; the knee, once measured, cannot be a budget artifact
RUN_STUDY=False: 12 runs gated; the frozen schedule above is the deliverable


**Interpretation.** The frozen schedule shows the price asymmetry doing its work: the
pure on-policy arm affords about 225,000 steps at the expensive per-step price, while the
75%-off arm packs about 366,000 cheap off-policy steps plus 56,000 on-policy ones, and the
verification loop confirms all four arms land within a single step's rounding of the same
3.0e9 token-pass budget. Both
accounting asserts passed, which retires the failure mode this design is most exposed to:
without the mechanical check, a one-line error in the split arithmetic would tilt the curve
and the knee would measure the bug, not the schedule.

What would decide it. H1 is supported if the measured curve rises and then flattens with the
knee, under the registered rule, strictly inside the sweep; Lab 07's cold-start finding
predicts exactly this shape, since warm-starting exists because early on-policy steps on a
cold student are spent teacher-scoring noise. The comparison to write in the report is the
knee's position against Lab 07's fixed warm-start heuristic: a knee near 25% would say a
light warm start suffices and the literature's sample-efficiency claim holds for most of the
budget; a knee at 75% or an ever-rising curve would say that at this scale the cheap-steps
argument dominates and on-policy training is a finishing step, not a regime. H0 (flat within
noise) is the negative result exercise 4 practices publishing: it would mean sequencing is a
free choice at this budget, which itself is actionable, choose whichever is operationally
simpler.

## Exercise 4: Publish the negative

**The exercise, restated.** Whatever H2 (the capstone's size-interaction hypothesis)
returned, write the two-paragraph result note you would post publicly, including the seed
ranges and the protocol hash. Negative results with auditable protocols are how a field
learns.

**The approach.** Since the capstone's runs are gated on this machine, the honest deliverable
is the note as a *template*: the full two-paragraph structure with every to-be-measured
number marked as an explicit placeholder in curly braces, so nothing in it can be mistaken
for a result that exists. This is a worked example answer, one defensible way to write such a
note, not the only one. The live check is the same kind the lab applied to its report
skeleton: a checklist assert that every element the note is required to carry (protocol hash,
seed count and list, primary metric, per-cell means with seed ranges, the registered decision
rule, the minimum detectable effect, limitations, and a reproduction pointer) is actually
present in the template, so the template cannot silently drop the part that makes a negative
result auditable rather than merely disappointing. The template is written to
`../runs/lab12/result_note_template.md` next to the protocols it cites.

In [5]:
NOTE = (
"## Result note: no resolvable size interaction in off- vs on-policy distillation\n"
"\n"
"We pre-registered (protocol {PROTOCOL_HASH}, frozen before any run) the hypothesis that\n"
"on-policy distillation's advantage over off-policy cached-logit distillation would be\n"
"larger for a smaller student (H2), across {N_SEEDS} seeds ({SEED_LIST}) and four arms at\n"
"matched token-pass compute. The primary metric, chosen in advance, was {PRIMARY_METRIC}.\n"
"Per-cell results (mean, with min-to-max seed range): {CELL_MEANS_AND_RANGES}. Under the\n"
"registered decision rule ({DECISION_RULE}), the measured interaction of {GAP} did not\n"
"exceed the minimum detectable effect of {MDE}, so the study returns: unresolved-negative.\n"
"We report this as 'no effect resolvable at this design', not 'no effect exists'.\n"
"\n"
"Limitations, registered in the protocol rather than discovered after: one model family\n"
"({MODEL_FAMILY}), one corpus domain ({CORPUS}), and compute matched by token-passes,\n"
"which counts arithmetic while ignoring that the two recipes stress hardware differently,\n"
"so equal token-passes need not mean equal wall-clock. A design that could resolve H2\n"
"would need either {N_SEEDS_NEEDED} seeds per cell (to shrink the noise floor below the\n"
"observed gap) or a larger student-size ratio between cells. All artifacts, manifests, and\n"
"the frozen protocol are at {ARTIFACT_URL}; every table regenerates from the manifest\n"
"chain via walk_manifests(), and the exact rerun commands are in the protocol's\n"
"reproduction section.\n")

REQUIRED = ["{PROTOCOL_HASH}", "{N_SEEDS}", "{SEED_LIST}", "{PRIMARY_METRIC}",
            "{CELL_MEANS_AND_RANGES}", "{DECISION_RULE}", "{GAP}", "{MDE}",
            "Limitations", "{ARTIFACT_URL}", "walk_manifests"]
missing = [r for r in REQUIRED if r not in NOTE]
assert not missing, f"template is missing required elements: {missing}"
placeholders = sorted(set(p for p in NOTE.split("{")[1:] for p in [p.split("}")[0]]))
open("../runs/lab12/result_note_template.md", "w").write(NOTE)
print(f"template written: ../runs/lab12/result_note_template.md")
print(f"checklist passed: all {len(REQUIRED)} required elements present")
print(f"placeholders a finished note must fill: {placeholders}")
assert len(NOTE.split(chr(10) + chr(10))) == 3, \
    "two paragraphs plus the title block, as the exercise specifies"
print("two-paragraph structure verified")


template written: ../runs/lab12/result_note_template.md
checklist passed: all 11 required elements present
placeholders a finished note must fill: ['ARTIFACT_URL', 'CELL_MEANS_AND_RANGES', 'CORPUS', 'DECISION_RULE', 'GAP', 'MDE', 'MODEL_FAMILY', 'N_SEEDS', 'N_SEEDS_NEEDED', 'PRIMARY_METRIC', 'PROTOCOL_HASH', 'SEED_LIST']
two-paragraph structure verified


**Interpretation.** The checklist assert passed with all eleven required elements
present, and the printed placeholder list is the note's honest interface: twelve named holes
that only executed runs can fill, so the template cannot leak invented numbers, which is the
solutions-spec honesty rule enforced by construction. Two choices in the wording deserve
defending, since this is one defensible answer rather than the answer. First, the verdict
vocabulary: "unresolved-negative" and the explicit sentence separating "no effect resolvable
at this design" from "no effect exists", because the most common failure of published
negative results is readers (and authors) collapsing that distinction; the note carries its
own minimum detectable effect precisely so a reader can judge what the study could ever have
seen. Second, the forward-looking sentence pricing the design that *could* resolve H2 (more
seeds or a wider size ratio): a negative result becomes useful to the field exactly when it
tells the next person what to run instead, and that sentence is generated from the same
MDE arithmetic the protocol registered, not from opinion.

With this, all four follow-on studies are designed, frozen, hashed, and power-checked, and
the notebook's `../runs/lab12/` directory now holds three new protocol files and one result
template beside the capstone's own. The gated work is scheduling; the auditable part, the
part the capstone says separates a practitioner's experiment from a study, exists and
verifies on this machine.